# PyPowSyBl / OpenLoadFlow AC Load-Flow Experiments

This notebook explores PyPowSyBl/OpenLoadFlow as an operational AC load-flow engine.

Main questions:

- How do we run Newton–Raphson AC load flow?
- What are the defaults?
- How do single and distributed slack differ?
- What happens when reactive limits are enabled?
- How do we extract bus, generator, branch, slack, and convergence results?


In [1]:
import json
import copy
import pandas as pd

import pypowsybl as pp
import pypowsybl.network as pn
import pypowsybl.loadflow as lf

print("pypowsybl version:", pp.__version__)
print("Available load-flow providers:", lf.get_provider_names())


pypowsybl version: 1.15.0
Available load-flow providers: ['DynaFlow', 'OpenLoadFlow']


## 1. Load a small test network

If no built-in factory works in your version, replace this cell with your repository network loader.


In [2]:

import pypowsybl as pp

print("Supported import formats:")
print(pp.network.get_import_formats())

print("\nSupported import extensions:")
print(pp.network.get_import_supported_extensions())

Supported import formats:
['BIIDM', 'CGMES', 'IEEE-CDF', 'JIIDM', 'MATPOWER', 'POWER-FACTORY', 'PSS/E', 'UCTE', 'XIIDM']

Supported import extensions:
['RAW', 'RAWX', 'UCT', 'biidm', 'bin', 'dgs', 'iidm', 'jiidm', 'json', 'mat', 'raw', 'rawx', 'txt', 'uct', 'xiidm', 'xml']


In [3]:

network = pp.network.load(r"C:\Users\Efthymis\OneDrive - University College Dublin\power-systems-tools-learning\data\raw\pglib\matpower_mat\pglib_opf_case1888_rte.mat")
#network = pp.network.create_ieee300()
#network = pp.network.create_ieee14()
print(network)


Network(id=pglib_opf_case1888_rte, name=pglib_opf_case1888_rte, case_date=2026-06-22 16:58:27.401000+00:00, forecast_distance=0:00:00, source_format=MATPOWER)


In [4]:
import pandas as pd

def safe_table(network, getter_name, *, display_table=True, max_rows=10):
    """
    Safely read a PyPowSyBl network table.

    Parameters
    ----------
    network : pypowsybl network object
    getter_name : str
        Example: "get_buses", "get_generators", "get_loads", "get_lines"
    display_table : bool
        Whether to display the table.
    max_rows : int
        Number of rows to display.

    Returns
    -------
    pd.DataFrame | None
    """
    if not hasattr(network, getter_name):
        print(f"{getter_name}: not available")
        return None

    try:
        df = getattr(network, getter_name)()
        print(f"\n{getter_name}: shape={df.shape}")
        print("columns:", list(df.columns))

        if display_table:
            display(df.head(max_rows))

        return df.copy()

    except Exception as exc:
        print(f"{getter_name}: failed -> {exc}")
        return None


def snapshot_network_tables(network, *, display_table=False):
    """
    Take a snapshot of the main network tables.

    Use this before and after load flow.
    """
    return {
        "buses": safe_table(network, "get_buses", display_table=display_table),
        "generators": safe_table(network, "get_generators", display_table=display_table),
        "loads": safe_table(network, "get_loads", display_table=display_table),
        "lines": safe_table(network, "get_lines", display_table=display_table),
    }


def compare_table_columns(before, after, table_name, columns):
    """
    Compare selected columns of one table before and after load flow.
    """
    before_df = before.get(table_name)
    after_df = after.get(table_name)

    print(f"\n=== {table_name.upper()} comparison ===")

    if before_df is None or after_df is None:
        print("Missing table before or after.")
        return None

    common_columns = [
        col for col in columns
        if col in before_df.columns and col in after_df.columns
    ]

    if not common_columns:
        print("None of the requested columns exist.")
        print("Available before:", list(before_df.columns))
        print("Available after:", list(after_df.columns))
        return None

    common_index = before_df.index.intersection(after_df.index)

    before_sel = before_df.loc[common_index, common_columns]
    after_sel = after_df.loc[common_index, common_columns]

    diff = after_sel.subtract(before_sel, fill_value=0)

    display(diff)

    numeric_diff = diff.select_dtypes(include="number")
    if not numeric_diff.empty:
        print("\nMax absolute difference:")
        display(numeric_diff.abs().max().sort_values(ascending=False))

    return diff


def compare_loadflow_state(before, after):
    """
    Compare the columns that usually contain solved load-flow quantities.

    Static data such as r, x, ratings, target_p, target_v, etc. should not
    necessarily change.
    """
    comparisons = {}

    comparisons["buses"] = compare_table_columns(
        before,
        after,
        "buses",
        ["v_mag", "v_angle"]
    )

    comparisons["generators"] = compare_table_columns(
        before,
        after,
        "generators",
        ["p", "q", "target_p", "target_q", "target_v"]
    )

    comparisons["loads"] = compare_table_columns(
        before,
        after,
        "loads",
        ["p", "q", "p0", "q0"]
    )

    comparisons["lines"] = compare_table_columns(
        before,
        after,
        "lines",
        ["p1", "q1", "i1", "p2", "q2", "i2"]
    )

    return comparisons

before = snapshot_network_tables(network, display_table=True)



get_buses: shape=(1888, 6)
columns: ['name', 'v_mag', 'v_angle', 'connected_component', 'synchronous_component', 'voltage_level_id']


,name,v_mag,v_angle,connected_component,synchronous_component,voltage_level_id
id,,,,,,
VL-1_0,,0.98889,0.0,0,0,VL-1
VL-2_0,,1.07552,0.0,0,0,VL-2
VL-3_0,,1.04444,0.0,0,0,VL-3
VL-4_0,,1.00334,0.0,0,0,VL-4
VL-4_1,,1.01111,0.0,0,0,VL-4
VL-4_2,,1.07064,0.0,0,0,VL-4
VL-5_0,,1.01111,0.0,0,0,VL-5
VL-5_1,,1.06522,0.0,0,0,VL-5
VL-7_0,,1.07237,0.0,0,0,VL-7



get_generators: shape=(297, 19)
columns: ['name', 'energy_source', 'target_p', 'min_p', 'max_p', 'min_q', 'max_q', 'rated_s', 'reactive_limits_kind', 'target_v', 'target_q', 'voltage_regulator_on', 'regulated_element_id', 'p', 'q', 'i', 'voltage_level_id', 'bus_id', 'connected']


,name,energy_source,target_p,min_p,max_p,min_q,max_q,rated_s,reactive_limits_kind,target_v,target_q,voltage_regulator_on,regulated_element_id,p,q,i,voltage_level_id,bus_id,connected
id,,,,,,,,,,,,,,,,,,,
GEN-14,,OTHER,16.56,5.52,27.6,0.0,0.0,100.0,MIN_MAX,1.016260,0.0,True,GEN-14,NaN,NaN,NaN,VL-14,VL-14_0,True
GEN-14#0,,OTHER,14.40,4.80,24.0,0.0,0.0,100.0,MIN_MAX,1.016260,0.0,True,GEN-14#0,NaN,NaN,NaN,VL-14,VL-14_0,True
GEN-46,,OTHER,18.00,6.00,30.0,-4.0,10.0,100.0,MIN_MAX,1.004215,3.0,True,GEN-46,NaN,NaN,NaN,VL-46,VL-46_0,True
GEN-58,,OTHER,0.00,1.20,6.0,1.9,2.0,100.0,MIN_MAX,1.004310,0.0,True,GEN-58,NaN,NaN,NaN,VL-58,,False
GEN-82,,OTHER,15.18,5.06,25.3,0.0,0.0,100.0,MIN_MAX,1.015620,0.0,True,GEN-82,NaN,NaN,NaN,VL-82,VL-82_0,True
GEN-89,,OTHER,15.00,5.00,25.0,-9.0,13.0,100.0,MIN_MAX,1.000000,2.0,True,GEN-89,NaN,NaN,NaN,VL-89,VL-89_0,True
GEN-100,,OTHER,78.00,26.00,130.0,-26.0,59.0,100.0,MIN_MAX,1.016470,16.5,True,GEN-100,NaN,NaN,NaN,VL-100,VL-100_0,True
GEN-125,,OTHER,14.04,4.68,23.4,-17.0,8.0,100.0,MIN_MAX,1.012380,-4.5,True,GEN-125,NaN,NaN,NaN,VL-125,VL-125_0,True
GEN-169,,OTHER,4.98,1.66,8.3,-8.3,8.3,100.0,MIN_MAX,1.000000,0.0,True,GEN-169,NaN,NaN,NaN,VL-169,VL-169_0,True



get_loads: shape=(1000, 10)
columns: ['name', 'type', 'p0', 'q0', 'p', 'q', 'i', 'voltage_level_id', 'bus_id', 'connected']


,name,type,p0,q0,p,q,i,voltage_level_id,bus_id,connected
id,,,,,,,,,,
LOAD-1,,UNDEFINED,5.8,-5.8,NaN,NaN,NaN,VL-1,VL-1_0,True
LOAD-3,,UNDEFINED,142.5,-13.4,NaN,NaN,NaN,VL-3,VL-3_0,True
LOAD-5,,UNDEFINED,32.8,-5.3,NaN,NaN,NaN,VL-5,VL-5_0,True
LOAD-6,,UNDEFINED,40.2,4.0,NaN,NaN,NaN,VL-4,VL-4_1,True
LOAD-10,,UNDEFINED,148.8,14.9,NaN,NaN,NaN,VL-10,VL-10_0,True
LOAD-11,,UNDEFINED,25.6,5.0,NaN,NaN,NaN,VL-11,VL-11_0,True
LOAD-12,,UNDEFINED,39.1,8.7,NaN,NaN,NaN,VL-12,VL-12_0,True
LOAD-13,,UNDEFINED,70.5,17.2,NaN,NaN,NaN,VL-13,VL-13_0,True
LOAD-14,,UNDEFINED,36.0,-4.2,NaN,NaN,NaN,VL-14,VL-14_0,True



get_lines: shape=(1976, 19)
columns: ['name', 'r', 'x', 'g1', 'b1', 'g2', 'b2', 'p1', 'q1', 'i1', 'p2', 'q2', 'i2', 'voltage_level1_id', 'voltage_level2_id', 'bus1_id', 'bus2_id', 'connected1', 'connected2']


,name,r,x,g1,b1,g2,b2,p1,q1,i1,p2,q2,i2,voltage_level1_id,voltage_level2_id,bus1_id,bus2_id,connected1,connected2
id,,,,,,,,,,,,,,,,,,,
LINE-1833-1,,2.607000e-05,0.000175,0.0,1.620,0.0,1.620,NaN,NaN,NaN,NaN,NaN,NaN,VL-709,VL-1,VL-709_0,VL-1_0,True,True
LINE-29-2,,2.100000e-07,0.000001,0.0,6.930,0.0,6.930,NaN,NaN,NaN,NaN,NaN,NaN,VL-29,VL-2,VL-29_0,VL-2_0,True,True
LINE-675-2,,1.450000e-06,0.000019,0.0,3.175,0.0,3.175,NaN,NaN,NaN,NaN,NaN,NaN,VL-674,VL-2,VL-674_1,VL-2_0,True,True
LINE-1271-2,,3.190000e-06,0.000032,0.0,4.190,0.0,4.190,NaN,NaN,NaN,NaN,NaN,NaN,VL-1269,VL-2,VL-1269_2,VL-2_0,True,True
LINE-30-2,,2.100000e-07,0.000001,0.0,6.930,0.0,6.930,NaN,NaN,NaN,NaN,NaN,NaN,VL-30,VL-2,VL-30_0,VL-2_0,True,True
LINE-142-3,,6.163000e-05,0.000411,0.0,3.795,0.0,3.795,NaN,NaN,NaN,NaN,NaN,NaN,VL-142,VL-3,VL-142_0,VL-3_0,True,True
LINE-677-3,,6.520000e-06,0.000044,0.0,0.405,0.0,0.405,NaN,NaN,NaN,NaN,NaN,NaN,VL-677,VL-3,VL-677_0,VL-3_0,True,True
LINE-1048-4,,9.733000e-05,0.000341,0.0,0.925,0.0,0.925,NaN,NaN,NaN,NaN,NaN,NaN,VL-1048,VL-4,VL-1048_0,VL-4_0,True,True
LINE-101-5,,2.770000e-06,0.000020,0.0,0.150,0.0,0.150,NaN,NaN,NaN,NaN,NaN,NaN,VL-101,VL-5,VL-101_0,VL-5_0,True,True


## 3. Helper for running AC load flow

In [5]:
def run_ac_case(network_in, name, parameters=None, provider="OpenLoadFlow"):
    net = copy.deepcopy(network_in)
    print(f"\n=== {name} ===")
    print("Provider:", provider)
    print("Parameters:", parameters)

    result = lf.run_ac(net, parameters=parameters, provider=provider)

    print("\nRaw result:")
    print(result)

    for i, r in enumerate(result):
        print(f"Component {i}:")
        for attr in [
            "status",
            "status_text",
            "iteration_count",
            "slack_bus_id",
            "slack_bus_active_power_mismatch",
            "distributed_active_power",
        ]:
            if hasattr(r, attr):
                print(f"  {attr}: {getattr(r, attr)}")

    return {
        "network": net,
        "result": result,
        "buses": safe_table(net, "get_buses"),
        "generators": safe_table(net, "get_generators"),
        "loads": safe_table(net, "get_loads"),
        "lines": safe_table(net, "get_lines"),
    }


In [6]:
baseline_params = lf.Parameters(
    voltage_init_mode=lf.VoltageInitMode.UNIFORM_VALUES,
    distributed_slack=False,
    use_reactive_limits=False,
    transformer_voltage_control_on=False,
    shunt_compensator_voltage_control_on=False,
    phase_shifter_regulation_on=False,
    provider_parameters={
        "acSolverType": "NEWTON_RAPHSON",
        "maxNewtonRaphsonIterations": '800'
    }
)


#print(results)

#after = snapshot_network_tables(network, display_table=True)

#diffs = compare_loadflow_state(before, after)


In [7]:
params_distributed = lf.Parameters(
    voltage_init_mode=lf.VoltageInitMode.UNIFORM_VALUES,
    distributed_slack=True,
    use_reactive_limits=False,
    transformer_voltage_control_on=False,
    shunt_compensator_voltage_control_on=False,
    phase_shifter_regulation_on=False,
    provider_parameters={
        "acSolverType": "NEWTON_RAPHSON",
        "maxNewtonRaphsonIterations": '800'
    }
)



## 6. Reactive-limit experiment

This enables reactive limits. To force PV/PQ switching, tighten one generator's Q limits in a copy of the network using your installed version's editing API.


In [8]:

params_q_limits = lf.Parameters(
    voltage_init_mode=lf.VoltageInitMode.UNIFORM_VALUES,
    distributed_slack=False,
    use_reactive_limits=True,
    transformer_voltage_control_on=False,
    shunt_compensator_voltage_control_on=False,
    phase_shifter_regulation_on=False,
    provider_parameters={
        "acSolverType": "NEWTON_RAPHSON",
        "reactiveLimitsMaxPqPvSwitch": "3",
        "maxNewtonRaphsonIterations": "800"
        
    }
)



## 7. Operational experiment 
Voltage Initialization with DC values, V=1pu. and using the calculated angles through DC pf
transformer_voltage_control_on=True,use_reactive_limits=True, phase_shifter_regulation_on=True

In [9]:
params_operational = lf.Parameters(
    voltage_init_mode=lf.VoltageInitMode.DC_VALUES,

    transformer_voltage_control_on=True,
    use_reactive_limits=True,
    phase_shifter_regulation_on=True,
    shunt_compensator_voltage_control_on=True,

    read_slack_bus=False,
    write_slack_bus=True,

    distributed_slack=True,
    balance_type=lf.BalanceType.PROPORTIONAL_TO_GENERATION_P_MAX,

    component_mode=lf.ComponentMode.MAIN_CONNECTED,

    hvdc_ac_emulation=True,

    provider_parameters={
        "acSolverType": "NEWTON_RAPHSON",

        "maxNewtonRaphsonIterations": '800'

    }
)


In [10]:
import copy
import time
import pandas as pd
import pypowsybl.loadflow as lf

def load_fresh_network():
    return pp.network.load(
        r"C:\Users\Efthymis\OneDrive - University College Dublin\power-systems-tools-learning\data\raw\pglib\matpower_mat\pglib_opf_case118_ieee.mat"
    )
def extract_component_metadata(results):
    rows = []

    for component_idx, result in enumerate(results):
        slack_bus_results = getattr(result, "slack_bus_results", None)

        slack_bus_id = None
        slack_active_power_mismatch = None

        if slack_bus_results:
            first_slack = slack_bus_results[0]
            slack_bus_id = getattr(first_slack, "id", None)
            slack_active_power_mismatch = getattr(first_slack, "active_power_mismatch", None)

        rows.append({
            "component_idx": component_idx,
            "status": getattr(result, "status", None),
            "status_text": getattr(result, "status_text", None),
            "inner_iterations": getattr(result, "iteration_count", None),
            "outer_loop_iterations": getattr(result, "outer_loop_iterations", None),
            "reference_bus_id": getattr(result, "reference_bus_id", None),
            "slack_bus_id": slack_bus_id,
            "slack_active_power_mismatch": slack_active_power_mismatch,
            "distributed_active_power": getattr(result, "distributed_active_power", None),
        })

    return rows


def run_lf_variant(
    base_network,
    name,
    params,
    provider="OpenLoadFlow",
    take_snapshots=False,
    verbose=True,
):
    """
    Run one AC load-flow variant and always return metadata,
    even if the solver fails or raises an exception.
    """

    if verbose:
        print("\n" + "=" * 80, flush=True)
        print(f"Starting load flow variant: {name}", flush=True)
        print(f"Provider: {provider}", flush=True)
        print(f"Parameters: {params}", flush=True)

    start = time.perf_counter()

    try:
        if verbose:
            print("Creating network copy...", flush=True)

        net = load_fresh_network()
        before = None
        after = None

        if take_snapshots:
            if verbose:
                print("Taking before snapshot...", flush=True)
            before = snapshot_network_tables(net, display_table=False)

        if verbose:
            print("Running AC load flow...", flush=True)

        results = lf.run_ac(
            net,
            parameters=params,
            provider=provider
        )

        elapsed = time.perf_counter() - start

        if verbose:
            print("Load flow finished.", flush=True)
            print(f"Elapsed time: {elapsed:.2f} s", flush=True)
            print("Raw result:", results, flush=True)

        if take_snapshots:
            if verbose:
                print("Taking after snapshot...", flush=True)
            after = snapshot_network_tables(net, display_table=False)

        component_metadata = extract_component_metadata(results)
        first = component_metadata[0] if component_metadata else {}

        metadata = {
            "name": name,
            "provider": provider,
            "exception": None,
            "elapsed_seconds": elapsed,

            "status": first.get("status"),
            "status_text": first.get("status_text"),

            "inner_iterations": first.get("inner_iterations"),
            "outer_loop_iterations": first.get("outer_loop_iterations"),

            "reference_bus_id": first.get("reference_bus_id"),
            "slack_bus_id": first.get("slack_bus_id"),
            "slack_active_power_mismatch": first.get("slack_active_power_mismatch"),
            "distributed_active_power": first.get("distributed_active_power"),
        }

        if verbose:
            print("Summary:")
            for k, v in metadata.items():
                print(f"  {k}: {v}", flush=True)

        return {
            "name": name,
            "network": net,
            "results": results,
            "before": before,
            "after": after,
            "component_metadata": component_metadata,
            "metadata": metadata,
        }

    except Exception as exc:
        elapsed = time.perf_counter() - start

        if verbose:
            print("Load flow raised an exception.", flush=True)
            print(f"Elapsed time before exception: {elapsed:.2f} s", flush=True)
            print(f"Exception: {type(exc).__name__}: {exc}", flush=True)

        metadata = {
            "name": name,
            "provider": provider,
            "exception": f"{type(exc).__name__}: {exc}",
            "elapsed_seconds": elapsed,

            "status": "EXCEPTION",
            "status_text": str(exc),

            "inner_iterations": None,
            "outer_loop_iterations": None,

            "reference_bus_id": None,
            "slack_bus_id": None,
            "slack_active_power_mismatch": None,
            "distributed_active_power": None,
        }

        return {
            "name": name,
            "network": None,
            "results": None,
            "before": None,
            "after": None,
            "component_metadata": [],
            "metadata": metadata,
        }

## 7. Initialization experiment

In [11]:
variants = {
    "fixed": ("fixed_controls_single_slack", baseline_params),
    "q_limits": ("reactive_limits_single_slack", params_q_limits),
    "distributed": ("distributed_slack", params_distributed),
    "operational": ("operational", params_operational),
}

runs = {}

for key, (name, params) in variants.items():
    runs[key] = run_lf_variant(
        network,
        name,
        params,
        provider="OpenLoadFlow",
        take_snapshots=False,   # keep False first for large 300-bus case
        verbose=True
    )
    
    
params_robust = lf.Parameters(
    voltage_init_mode=lf.VoltageInitMode.DC_VALUES,
    distributed_slack=True,
    balance_type=lf.BalanceType.PROPORTIONAL_TO_GENERATION_P_MAX,

    use_reactive_limits=True,

    transformer_voltage_control_on=False,
    shunt_compensator_voltage_control_on=False,
    phase_shifter_regulation_on=False,

    component_mode=lf.ComponentMode.MAIN_CONNECTED,

    provider_parameters={
        "acSolverType": "NEWTON_RAPHSON",
        "maxNewtonRaphsonIterations": "800",
        "slackBusSelectionMode": "LARGEST_GENERATOR"
    }
)

runs["robust"] = run_lf_variant(
    network,
    "robust_dc_init_distributed_slack",
    params_robust,
    take_snapshots=False,
    verbose=True
)


Starting load flow variant: fixed_controls_single_slack
Provider: OpenLoadFlow
Parameters: Parameters(voltage_init_mode=UNIFORM_VALUES, transformer_voltage_control_on=False, use_reactive_limits=False, phase_shifter_regulation_on=False, twt_split_shunt_admittance=False, shunt_compensator_voltage_control_on=False, read_slack_bus=True, write_slack_bus=True, distributed_slack=False, balance_type=PROPORTIONAL_TO_GENERATION_P_MAX, dc_use_transformer_ratio=True, countries_to_balance=[], component_mode=<ComponentMode.MAIN_CONNECTED: 0>, hvdc_ac_emulation=True, dc_power_factor=1.0, dc=False, provider_parameters={'acSolverType': 'NEWTON_RAPHSON', 'maxNewtonRaphsonIterations': '800'})
Creating network copy...
Running AC load flow...
Load flow finished.
Elapsed time: 0.11 s
Raw result: [ComponentResult(connected_component_num=0, synchronous_component_num=0, status=CONVERGED, status_text=Converged, iteration_count=3, reference_bus_id='VL-68_1', slack_bus_results=[SlackBusResult(id='VL-68_1', activ

In [ ]:
summary = pd.DataFrame([
    run["metadata"]
    for run in runs.values()
])

display(summary)

,name,provider,exception,elapsed_seconds,status,status_text,inner_iterations,outer_loop_iterations,reference_bus_id,slack_bus_id,slack_active_power_mismatch,distributed_active_power
0,fixed_controls_single_slack,OpenLoadFlow,None,0.107344,ComponentStatus.CONVERGED,Converged,3,None,VL-68_1,VL-68_1,1228.608983,0.000000
1,reactive_limits_single_slack,OpenLoadFlow,None,0.013514,ComponentStatus.CONVERGED,Converged,7,None,VL-68_1,VL-68_1,1229.752416,0.000000
2,distributed_slack,OpenLoadFlow,None,0.010960,ComponentStatus.CONVERGED,Converged,9,None,VL-68_1,VL-68_1,-0.138091,1125.916214
3,operational,OpenLoadFlow,None,0.015181,ComponentStatus.CONVERGED,Converged,12,None,VL-49_0,VL-49_0,-0.031450,1123.938556
4,robust_dc_init_distributed_slack,OpenLoadFlow,None,0.013688,ComponentStatus.CONVERGED,Converged,13,None,VL-68_1,VL-68_1,0.133478,1123.777240


In [13]:
gens = network.get_generators()
loads = network.get_loads()

print("Total target_p:", gens["target_p"].sum())
print("Total load p0:", loads["p0"].sum())
print("Generation - Load:", gens["target_p"].sum() - loads["p0"].sum())

display(gens[["target_p", "min_p", "max_p"]].describe())
display(loads[["p0", "q0"]].describe())

Total target_p: 57105.785
Total load p0: 59110.5
Generation - Load: -2004.7149999999965


,target_p,min_p,max_p
count,297.000000,297.000000,297.000000
mean,192.275370,85.865522,307.584209
std,267.938754,115.902614,424.631706
min,-0.000000,-8.000000,0.000000
25%,19.200000,6.400000,32.000000
50%,37.200000,12.200000,63.000000
75%,242.000000,156.000000,439.800000
max,953.000000,403.000000,1503.000000


,p0,q0
count,1000.000000,1000.000000
mean,59.110500,2.270900
std,74.523427,23.378685
min,-73.500000,-183.300000
25%,16.625000,-4.225000
50%,37.900000,1.600000
75%,80.100000,8.625000
max,710.500000,340.400000


In [14]:
params_robust = lf.Parameters(
    voltage_init_mode=lf.VoltageInitMode.DC_VALUES,
    distributed_slack=True,
    balance_type=lf.BalanceType.PROPORTIONAL_TO_GENERATION_P_MAX,

    use_reactive_limits=True,

    transformer_voltage_control_on=False,
    shunt_compensator_voltage_control_on=False,
    phase_shifter_regulation_on=False,

    component_mode=lf.ComponentMode.MAIN_CONNECTED,

    provider_parameters={
        "acSolverType": "NEWTON_RAPHSON",
        "maxNewtonRaphsonIterations": "800",
        "slackBusSelectionMode": "LARGEST_GENERATOR"
    }
)

runs["robust"] = run_lf_variant(
    network,
    "robust_dc_init_distributed_slack",
    params_robust,
    take_snapshots=False,
    verbose=True
)


Starting load flow variant: robust_dc_init_distributed_slack
Provider: OpenLoadFlow
Parameters: Parameters(voltage_init_mode=DC_VALUES, transformer_voltage_control_on=False, use_reactive_limits=True, phase_shifter_regulation_on=False, twt_split_shunt_admittance=False, shunt_compensator_voltage_control_on=False, read_slack_bus=True, write_slack_bus=True, distributed_slack=True, balance_type=PROPORTIONAL_TO_GENERATION_P_MAX, dc_use_transformer_ratio=True, countries_to_balance=[], component_mode=<ComponentMode.MAIN_CONNECTED: 0>, hvdc_ac_emulation=True, dc_power_factor=1.0, dc=False, provider_parameters={'acSolverType': 'NEWTON_RAPHSON', 'maxNewtonRaphsonIterations': '800', 'slackBusSelectionMode': 'LARGEST_GENERATOR'})
Creating network copy...
Running AC load flow...
Load flow finished.
Elapsed time: 0.01 s
Raw result: [ComponentResult(connected_component_num=0, synchronous_component_num=0, status=CONVERGED, status_text=Converged, iteration_count=13, reference_bus_id='VL-68_1', slack_bu

In [13]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, clear_output


PROJECT_ROOT = Path(r"C:\Users\Efthymis\OneDrive - University College Dublin\power-systems-tools-learning")

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from pypowsybl_replacement_note_experiments import (
    parse_replacement_notes,
    run_case_replacement_experiments,
    notebook_progress,
)

In [14]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()

if cwd.name == "python":
    repo_root = cwd.parent
else:
    repo_root = cwd

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.pypowsybl_replacement_experiment_matrix import (
    run_case_experiment_matrix,
    make_replacement_variants,
    make_parameter_scenarios,
    notebook_progress,
)

In [15]:
import pandas as pd
from pathlib import Path

variants = make_replacement_variants()
scenarios = make_parameter_scenarios()

case_names = [
    "pglib_opf_case5_pjm",
    "pglib_opf_case14_ieee",
    "pglib_opf_case24_ieee_rts",
    "pglib_opf_case30_ieee",
    "pglib_opf_case118_ieee",
    "pglib_opf_case300_ieee",
    "pglib_opf_case1888_rte",
    # "pglib_opf_case6470_rte",
    # "pglib_opf_case9241_pegase",
    # "pglib_opf_case13659_pegase",
]

replacement_variants=[
        variants["original"],
        variants["BV_from_previous"],          # warmstart only
        variants["G_PG_from"],
        variants["G_PGQG_from"],
        variants["G_PGQG_from_plus_GVG_from"],
    ],

selected_scenarios = [
    scenarios["strict"],
    scenarios["robust_fixed"],
]

all_summaries = []
all_audits = []

for case_name in case_names:
    print(f"\n\n### Running {case_name}")

    summary_i, audit_i = run_case_experiment_matrix(
        case_name,
        replacement_variants=selected_variants,
        parameter_scenarios=selected_scenarios,
        summary_csv=None,
        audit_csv=None,
        progress=notebook_progress,
    )

    all_summaries.append(summary_i)
    all_audits.append(audit_i)

summary_all = pd.concat(all_summaries, ignore_index=True)
audit_all = pd.concat(all_audits, ignore_index=True)

summary_all



### Running pglib_opf_case5_pjm
Starting pglib_opf_case5_pjm | original | strict

=== pglib_opf_case5_pjm | original | strict ===
status=ComponentStatus.CONVERGED iterations=2 balance_after=-235.0
Finished original | strict: status=ComponentStatus.CONVERGED iterations=2
Starting pglib_opf_case5_pjm | original | robust_fixed

=== pglib_opf_case5_pjm | original | robust_fixed ===
status=ComponentStatus.CONVERGED iterations=4 balance_after=-235.0
Finished original | robust_fixed: status=ComponentStatus.CONVERGED iterations=4
Starting pglib_opf_case5_pjm | G_PG_from | strict

=== pglib_opf_case5_pjm | G_PG_from | strict ===
status=ComponentStatus.CONVERGED iterations=2 balance_after=0.0
Finished G_PG_from | strict: status=ComponentStatus.CONVERGED iterations=2
Starting pglib_opf_case5_pjm | G_PG_from | robust_fixed

=== pglib_opf_case5_pjm | G_PG_from | robust_fixed ===
status=ComponentStatus.CONVERGED iterations=3 balance_after=0.0
Finished G_PG_from | robust_fixed: status=ComponentStat

,case_name,variant_name,parameter_scenario,replacement_groups,direction,mapping_success,n_bus_voltage_notes,n_gen_pgqg_notes,n_gen_vg_notes,n_load_notes,...,slack_bus_id,slack_active_power_mismatch,distributed_active_power,elapsed_seconds,exception,total_max_generation_after,total_headroom_after,number_of_generators_with_max_p_zero,number_of_generators_with_target_p_above_max_p,number_of_generators_with_target_q_outside_limits
0,pglib_opf_case5_pjm,original,strict,,NaN,True,5,5,5,0,...,VL-4_0,2.377418e+02,0.000000,0.002795,None,1530.00,765.000,0.0,0.0,0.0
1,pglib_opf_case5_pjm,original,robust_fixed,,NaN,True,5,5,5,0,...,VL-4_0,6.232218e-01,237.742512,0.002479,None,1530.00,765.000,0.0,0.0,0.0
2,pglib_opf_case5_pjm,G_PG_from,strict,G_PG,from,True,5,5,5,0,...,VL-4_0,5.026169e+00,0.000000,0.002313,None,1530.00,530.000,0.0,0.0,0.0
3,pglib_opf_case5_pjm,G_PG_from,robust_fixed,G_PG,from,True,5,5,5,0,...,VL-4_0,3.455662e-02,5.027162,0.001798,None,1530.00,530.000,0.0,0.0,0.0
4,pglib_opf_case5_pjm,G_PGQG_from,strict,G_PG+G_QG,from,True,5,5,5,0,...,VL-4_0,5.026169e+00,0.000000,0.001850,None,1530.00,530.000,0.0,0.0,0.0
5,pglib_opf_case5_pjm,G_PGQG_from,robust_fixed,G_PG+G_QG,from,True,5,5,5,0,...,VL-4_0,3.455662e-02,5.027162,0.001603,None,1530.00,530.000,0.0,0.0,0.0
6,pglib_opf_case5_pjm,G_PGQG_from_plus_GVG_from,strict,G_PG+G_QG+G_VG,from,True,5,5,5,0,...,VL-4_0,5.026169e+00,0.000000,0.002563,None,1530.00,530.000,0.0,0.0,0.0
7,pglib_opf_case5_pjm,G_PGQG_from_plus_GVG_from,robust_fixed,G_PG+G_QG+G_VG,from,True,5,5,5,0,...,VL-4_0,3.455662e-02,5.027162,0.001647,None,1530.00,530.000,0.0,0.0,0.0
8,pglib_opf_case14_ieee,original,strict,,NaN,True,14,5,5,0,...,VL-1_0,7.616580e+01,0.000000,0.001426,None,399.00,199.500,3.0,0.0,0.0
9,pglib_opf_case14_ieee,original,robust_fixed,,NaN,True,14,5,5,0,...,VL-1_0,-7.818840e-01,76.163395,0.001926,None,399.00,199.500,3.0,0.0,0.0


In [16]:
summary_all.pivot_table(
    index=["case_name", "variant_name"],
    columns="parameter_scenario",
    values="status",
    aggfunc="first",
)

parameter_scenario                                                robust_fixed  \
case_name                 variant_name                                           
pglib_opf_case118_ieee    G_PGQG_from                ComponentStatus.CONVERGED   
                          G_PGQG_from_plus_GVG_from  ComponentStatus.CONVERGED   
                          G_PG_from                  ComponentStatus.CONVERGED   
                          original                   ComponentStatus.CONVERGED   
pglib_opf_case14_ieee     G_PGQG_from                ComponentStatus.CONVERGED   
                          G_PGQG_from_plus_GVG_from  ComponentStatus.CONVERGED   
                          G_PG_from                  ComponentStatus.CONVERGED   
                          original                   ComponentStatus.CONVERGED   
pglib_opf_case1888_rte    G_PGQG_from                ComponentStatus.CONVERGED   
                          G_PGQG_from_plus_GVG_from                  EXCEPTION   
                          G_PG_from                  ComponentStatus.CONVERGED   
                          original                   ComponentStatus.CONVERGED   
pglib_opf_case24_ieee_rts G_PGQG_from                ComponentStatus.CONVERGED   
                          G_PGQG_from_plus_GVG_from  ComponentStatus.CONVERGED   
                          G_PG_from                  ComponentStatus.CONVERGED   
                          original                   ComponentStatus.CONVERGED   
pglib_opf_case300_ieee    G_PGQG_from                ComponentStatus.CONVERGED   
                          G_PGQG_from_plus_GVG_from  ComponentStatus.CONVERGED   
                          G_PG_from                  ComponentStatus.CONVERGED   
                          original                      ComponentStatus.FAILED   
pglib_opf_case30_ieee     G_PGQG_from                ComponentStatus.CONVERGED   
                          G_PGQG_from_plus_GVG_from  ComponentStatus.CONVERGED   
                          G_PG_from                  ComponentStatus.CONVERGED   
                          original                   ComponentStatus.CONVERGED   
pglib_opf_case5_pjm       G_PGQG_from                ComponentStatus.CONVERGED   
                          G_PGQG_from_plus_GVG_from  ComponentStatus.CONVERGED   
                          G_PG_from                  ComponentStatus.CONVERGED   
                          original                   ComponentStatus.CONVERGED   

parameter_scenario                                                      strict  
case_name                 variant_name                                          
pglib_opf_case118_ieee    G_PGQG_from                ComponentStatus.CONVERGED  
                          G_PGQG_from_plus_GVG_from  ComponentStatus.CONVERGED  
                          G_PG_from                  ComponentStatus.CONVERGED  
                          original                   ComponentStatus.CONVERGED  
pglib_opf_case14_ieee     G_PGQG_from                ComponentStatus.CONVERGED  
                          G_PGQG_from_plus_GVG_from  ComponentStatus.CONVERGED  
                          G_PG_from                  ComponentStatus.CONVERGED  
                          original                   ComponentStatus.CONVERGED  
pglib_opf_case1888_rte    G_PGQG_from                   ComponentStatus.FAILED  
                          G_PGQG_from_plus_GVG_from                  EXCEPTION  
                          G_PG_from                     ComponentStatus.FAILED  
                          original                      ComponentStatus.FAILED  
pglib_opf_case24_ieee_rts G_PGQG_from                ComponentStatus.CONVERGED  
                          G_PGQG_from_plus_GVG_from  ComponentStatus.CONVERGED  
                          G_PG_from                  ComponentStatus.CONVERGED  
                          original                   ComponentStatus.CONVERGED  
pglib_opf_case300_ieee    G_PGQG_from                ComponentStatus.CONVERGED  


In [ ]:
results_dir = Path(repo_root) / "results"

summary_all.to_csv(
    results_dir / "pypowsybl_multi_case_replacement_summary.csv",
    index=False,
)

audit_all.to_csv(
    results_dir / "pypowsybl_multi_case_replacement_audit.csv",
    index=False,
)

KeyError: 'reactive_limits'

In [10]:
variants = make_replacement_variants()
scenarios = make_parameter_scenarios()

summary, audit = run_case_experiment_matrix(
    "pglib_opf_case13659_pegase.m",
    replacement_variants=[
        variants["original"],
        variants["G_PG_from"],
        variants["G_PGQG_from"],
        variants["G_PGQG_from_plus_GVG_from"],
    ],
    parameter_scenarios=[
        scenarios["robust_fixed"],
        scenarios["reactive_limits"],
    ],
    progress=notebook_progress,
)
summary

KeyError: 'reactive_limits'